# Gold Layer DDL — Star Schema Tables and Views

## Overview
This Databricks notebook implements the **Gold Layer** of the Medallion Architecture data warehouse.
The Gold layer represents the business-level analytical data model organized as a **Star Schema**.

It transforms, conforms, and enriches data from the **Silver Layer** into dimensional and fact entities:

- **Dimensions**:
  1. `gold.dim_customers` — Conformed customer dimension combining CRM customer profiles with ERP demographics and geographic data. Implements Master Data Management (MDM) gender precedence and assigns surrogate key `customer_key`.
  2. `gold.dim_products` — Conformed product dimension combining CRM product catalog with ERP category hierarchy. Filters out historical SCD versions (`WHERE prd_end_dt IS NULL`), projects both `cost` and `product_cost` for 100% downstream compatibility, and assigns surrogate key `product_key`.

- **Facts**:
  3. `gold.fact_sales` — Central transactional sales fact table joined to product and customer dimensions to resolve foreign surrogate keys (`product_key`, `customer_key`).

All tables are stored in **Delta Lake** format (`USING DELTA`), enabling ACID guarantees, time travel, and high-performance querying for BI tools and downstream analytics.

In [ ]:
# Databricks notebook source
# COMMAND ----------
# Helper display function fallback for standalone PySpark / test runner environments
try:
    display
except NameError:
    def display(df, n=20):
        if hasattr(df, "show"):
            df.show(n, truncate=False)
        else:
            print(df)

# Ensure gold database exists
spark.sql("CREATE DATABASE IF NOT EXISTS gold")
print("Database 'gold' verified/created successfully.")

## Dimension: `gold.dim_customers`

### Business Logic & Transformations:
- **Surrogate Key Generation**: Sequential surrogate key `customer_key` generated via `ROW_NUMBER() OVER (ORDER BY ci.cst_id)`.
- **Master Data Management (MDM)**: CRM is the authoritative master system for gender (`ci.cst_gndr != 'unknown'`). If unknown, falls back to ERP demographic records (`ca.gen`).
- **Enrichment**: Joins CRM customers (`silver.crm_cust_info`) with ERP demographics (`silver.erp_cust_az12`) and ERP locations (`silver.erp_loc_a101`) via standardized customer natural key (`cst_key = cid`).

In [ ]:
# COMMAND ----------
# Create or replace Delta table: gold.dim_customers
spark.sql("DROP TABLE IF EXISTS gold.dim_customers")
spark.sql("""
CREATE TABLE gold.dim_customers
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY ci.cst_id) AS customer_key,
    ci.cst_id AS customer_id,
    ci.cst_key AS customer_number, 
    ci.cst_firstname AS first_name, 
    ci.cst_lastname AS last_name, 
    la.cntry AS country,
    ci.cst_marital_status AS marital_status, 
    CASE WHEN ci.cst_gndr != 'unknown' THEN ci.cst_gndr
         ELSE COALESCE(ca.gen, 'unknown')
    END AS gender,
    ca.bdate AS birthdate,
    ci.cst_create_date AS create_date
FROM silver.crm_cust_info AS ci
LEFT JOIN silver.erp_cust_az12 AS ca
    ON ci.cst_key = ca.cid
LEFT JOIN silver.erp_loc_a101 AS la
    ON ci.cst_key = la.cid
""")
print(">> Table created: gold.dim_customers")

In [ ]:
# COMMAND ----------
# PySpark DataFrame equivalent with surrogate key window and MDM gender resolution
from pyspark.sql.functions import col, when, coalesce, lit, row_number
from pyspark.sql.window import Window

w_cust = Window.orderBy("cst_id")
ci = spark.table("silver.crm_cust_info")
ca = spark.table("silver.erp_cust_az12")
la = spark.table("silver.erp_loc_a101")

df_dim_customers = (
    ci
    .join(ca, ci.cst_key == ca.cid, "left")
    .join(la, ci.cst_key == la.cid, "left")
    .select(
        ci.cst_id.alias("customer_id"),
        ci.cst_key.alias("customer_number"),
        ci.cst_firstname.alias("first_name"),
        ci.cst_lastname.alias("last_name"),
        la.cntry.alias("country"),
        ci.cst_marital_status.alias("marital_status"),
        when(ci.cst_gndr != "unknown", ci.cst_gndr)
            .otherwise(coalesce(ca.gen, lit("unknown"))).alias("gender"),
        ca.bdate.alias("birthdate"),
        ci.cst_create_date.alias("create_date")
    )
    .withColumn("customer_key", row_number().over(w_cust))
    .select(
        "customer_key",
        "customer_id",
        "customer_number",
        "first_name",
        "last_name",
        "country",
        "marital_status",
        "gender",
        "birthdate",
        "create_date"
    )
)

# Display sample records from gold.dim_customers
df_sample_cust = spark.table("gold.dim_customers")
display(df_sample_cust.limit(10))

## Dimension: `gold.dim_products`

### Business Logic & Transformations:
- **Surrogate Key Generation**: Sequential surrogate key `product_key` generated via `ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key)`.
- **Active SCD Record Filter**: `WHERE pn.prd_end_dt IS NULL` filters out all expired historical records, retaining only the current active version of each product.
- **Dual Cost Column Projection**: Projects both `pn.prd_cost AS cost` and `pn.prd_cost AS product_cost` to maintain 100% downstream compatibility with the original DDL specification and all 14 analytics queries.
- **Enrichment**: Enriches CRM product records (`silver.crm_prd_info`) with ERP category hierarchy (`silver.erp_px_cat_g1v2`) on `pn.cat_id = pc.id`.

In [ ]:
# COMMAND ----------
# Create or replace Delta table: gold.dim_products
spark.sql("DROP TABLE IF EXISTS gold.dim_products")
spark.sql("""
CREATE TABLE gold.dim_products
USING DELTA AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key,
    pn.prd_id AS product_id,
    pn.prd_key AS product_number,
    pn.prd_nm AS product_name,
    pn.cat_id AS category_id,
    pc.cat AS category,
    pc.subcat AS subcategory,
    pc.maintenance AS maintenance,
    pn.prd_cost AS cost,
    pn.prd_cost AS product_cost,
    pn.prd_line AS product_line,
    pn.prd_start_dt AS start_date
FROM silver.crm_prd_info AS pn
LEFT JOIN silver.erp_px_cat_g1v2 AS pc
    ON pn.cat_id = pc.id
WHERE pn.prd_end_dt IS NULL
""")
print(">> Table created: gold.dim_products")

In [ ]:
# COMMAND ----------
# PySpark DataFrame equivalent with SCD active filter and dual cost projection
w_prd = Window.orderBy("prd_start_dt", "prd_key")
pn = spark.table("silver.crm_prd_info")
pc = spark.table("silver.erp_px_cat_g1v2")

df_dim_products = (
    pn.filter(pn.prd_end_dt.isNull())
    .join(pc, pn.cat_id == pc.id, "left")
    .select(
        pn.prd_id.alias("product_id"),
        pn.prd_key.alias("product_number"),
        pn.prd_nm.alias("product_name"),
        pn.cat_id.alias("category_id"),
        pc.cat.alias("category"),
        pc.subcat.alias("subcategory"),
        pc.maintenance.alias("maintenance"),
        pn.prd_cost.alias("cost"),
        pn.prd_cost.alias("product_cost"),
        pn.prd_line.alias("product_line"),
        pn.prd_start_dt.alias("start_date")
    )
    .withColumn("product_key", row_number().over(w_prd))
    .select(
        "product_key",
        "product_id",
        "product_number",
        "product_name",
        "category_id",
        "category",
        "subcategory",
        "maintenance",
        "cost",
        "product_cost",
        "product_line",
        "start_date"
    )
)

# Display sample records from gold.dim_products
df_sample_prd = spark.table("gold.dim_products")
display(df_sample_prd.limit(10))

## Fact Table: `gold.fact_sales`

### Business Logic & Transformations:
- **Central Fact Model**: Represents transactional sales line items conformed to dimensional grain.
- **Surrogate Key Resolution**: Joins `silver.crm_sales_details` with `gold.dim_products` on product natural key (`sd.sls_prd_key = pr.product_number`) and with `gold.dim_customers` on customer ID (`sd.sls_cust_id = cu.customer_id`) to attach surrogate keys `product_key` and `customer_key`.
- **Fact Measures**: `sales_amount`, `quantity`, `price`.
- **Fact Dimensions**: `order_number`, `order_date`, `shipping_date`, `due_date`, `customer_key`, `product_key`.

In [ ]:
# COMMAND ----------
# Create or replace Delta table: gold.fact_sales
spark.sql("DROP TABLE IF EXISTS gold.fact_sales")
spark.sql("""
CREATE TABLE gold.fact_sales
USING DELTA AS
SELECT 
    sd.sls_ord_num AS order_number,
    pr.product_key,
    cu.customer_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt AS shipping_date,
    sd.sls_due_dt AS due_date,
    sd.sls_sales AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price AS price
FROM silver.crm_sales_details AS sd 
JOIN gold.dim_products AS pr
    ON sd.sls_prd_key = pr.product_number
JOIN gold.dim_customers AS cu
    ON sd.sls_cust_id = cu.customer_id
""")
print(">> Table created: gold.fact_sales")

In [ ]:
# COMMAND ----------
# PySpark DataFrame equivalent with surrogate foreign key joins
sd = spark.table("silver.crm_sales_details")
dim_p = spark.table("gold.dim_products")
dim_c = spark.table("gold.dim_customers")

df_fact_sales = (
    sd
    .join(dim_p, sd.sls_prd_key == dim_p.product_number, "inner")
    .join(dim_c, sd.sls_cust_id == dim_c.customer_id, "inner")
    .select(
        sd.sls_ord_num.alias("order_number"),
        dim_p.product_key,
        dim_c.customer_key,
        sd.sls_order_dt.alias("order_date"),
        sd.sls_ship_dt.alias("shipping_date"),
        sd.sls_due_dt.alias("due_date"),
        sd.sls_sales.alias("sales_amount"),
        sd.sls_quantity.alias("quantity"),
        sd.sls_price.alias("price")
    )
)

# Display sample records from gold.fact_sales
df_sample_sales = spark.table("gold.fact_sales")
display(df_sample_sales.limit(10))

## Gold Layer View Definitions (SQL View Compatibility)

For workflows where virtual views are preferred over materialized Delta tables, the views below provide identical logic matching the original `ddl_gold.sql` script.

In [ ]:
# COMMAND ----------
# 1. View: gold.v_dim_customers
spark.sql("""
CREATE OR REPLACE VIEW gold.v_dim_customers AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY ci.cst_id) AS customer_key,
    ci.cst_id AS customer_id,
    ci.cst_key AS customer_number, 
    ci.cst_firstname AS first_name, 
    ci.cst_lastname AS last_name, 
    la.cntry AS country,
    ci.cst_marital_status AS marital_status, 
    CASE WHEN ci.cst_gndr != 'unknown' THEN ci.cst_gndr
         ELSE COALESCE(ca.gen, 'unknown')
    END AS gender,
    ca.bdate AS birthdate,
    ci.cst_create_date AS create_date
FROM silver.crm_cust_info AS ci
LEFT JOIN silver.erp_cust_az12 AS ca
    ON ci.cst_key = ca.cid
LEFT JOIN silver.erp_loc_a101 AS la
    ON ci.cst_key = la.cid
""")

# 2. View: gold.v_dim_products
spark.sql("""
CREATE OR REPLACE VIEW gold.v_dim_products AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key,
    pn.prd_id AS product_id,
    pn.prd_key AS product_number,
    pn.prd_nm AS product_name,
    pn.cat_id AS category_id,
    pc.cat AS category,
    pc.subcat AS subcategory,
    pc.maintenance AS maintenance,
    pn.prd_cost AS cost,
    pn.prd_cost AS product_cost,
    pn.prd_line AS product_line,
    pn.prd_start_dt AS start_date
FROM silver.crm_prd_info AS pn
LEFT JOIN silver.erp_px_cat_g1v2 AS pc
    ON pn.cat_id = pc.id
WHERE pn.prd_end_dt IS NULL
""")

# 3. View: gold.v_fact_sales
spark.sql("""
CREATE OR REPLACE VIEW gold.v_fact_sales AS
SELECT 
    sd.sls_ord_num AS order_number,
    pr.product_key,
    cu.customer_key,
    sd.sls_order_dt AS order_date,
    sd.sls_ship_dt AS shipping_date,
    sd.sls_due_dt AS due_date,
    sd.sls_sales AS sales_amount,
    sd.sls_quantity AS quantity,
    sd.sls_price AS price
FROM silver.crm_sales_details AS sd 
JOIN gold.dim_products AS pr
    ON sd.sls_prd_key = pr.product_number
JOIN gold.dim_customers AS cu
    ON sd.sls_cust_id = cu.customer_id
""")
print(">> Views created: gold.v_dim_customers, gold.v_dim_products, gold.v_fact_sales")

## Verification & Catalog Inspection

Verify the registered tables and views in the `gold` schema and display their detailed column schemas.

In [ ]:
# COMMAND ----------
# List all tables and views in the gold database
df_gold_tables = spark.sql("SHOW TABLES IN gold")
display(df_gold_tables)

# Inspect schemas for each core Star Schema table
for tbl in ["gold.dim_customers", "gold.dim_products", "gold.fact_sales"]:
    print(f"\n--- Schema Definition: {tbl} ---")
    spark.sql(f"DESCRIBE TABLE {tbl}").show(truncate=False)